In [ ]:
!pip install cyipopt
!pip install qhdopt
!pip install gurobipy
!pip install pandas
!pip install numpy>=1.26.4
!pip install attrs
!pip install optax
!pip install filelock
!pip install nbconvert
!pip install jax[cpu]
!pip install mpax>=0.2.4
!pip install scipy>=1.12.0 --upgrade

In [ ]:
!git clone 'https://github.com/Artephi-Computing/OpenPhiSolve.git'

In [ ]:
import os
os.chdir('OpenPhiSolve')

In [ ]:
!ls -F

'=1.26.4'   examples/   phisolve/	     requirements.txt
 build/     img/        phisolve.egg-info/   setup.py
 check/     LICENSE     README.md	     tox.ini


In [ ]:
!pip install .

In [ ]:
import gurobipy as gp
from gurobipy import Model, GRB

import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import os
import math
import time

from collections import defaultdict
from scipy.sparse import coo_matrix, csr_matrix, hstack, vstack, eye

from qhdopt import QHD
from sympy import symbols, exp

from phisolve import PhiMIQP, MIQP, QIHD, PDQP
import jax
from phisolve.utils.jax_utils import jax_device

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
#print(os.listdir("/content"))
path = "/content/OpenPhiSolve"
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

print(csv_files)

In [ ]:
# List of your 5 structures
structures = ["Bladder", "Femur_Head_L", "Femur_Head_R", "Prostate", "Rectum"]

for struct in structures:
    # 1. Load the files (skiprows=1 drops the beamlet-name header row)
    df_l = pd.read_csv(f"PROSTATE___90___{struct}.csv",  skiprows=1, header=None)
    df_r = pd.read_csv(f"PROSTATE___270___{struct}.csv", skiprows=1, header=None)

    # Sanity check: both angles must cover the same voxels for hstack to be meaningful
    assert df_l.shape[0] == df_r.shape[0], (
        f"{struct}: voxel count mismatch between 90* ({df_l.shape[0]}) and 270* ({df_r.shape[0]})"
    )

    print(f"Structure: {struct}")
    print(f"  LLat Original: {df_l.shape[0]} voxels, {df_l.shape[1]-4} beamlet columns")
    print(f"  RLat Original: {df_r.shape[0]} voxels, {df_r.shape[1]-4} beamlet columns")

    # 2. Drop the first four metadata columns (voxel_id, x, y, z), keep dose values
    mat_l = df_l.iloc[:, 4:].to_numpy(dtype=float)
    mat_r = df_r.iloc[:, 4:].to_numpy(dtype=float)

    # 3. Horizontally concatenate: voxel i now sees both the 90* and 270* beamlets
    combined_matrix = np.hstack((mat_l, mat_r))

    # 4. Convert to sparse COO (row, col, value triplets)
    coo = coo_matrix(combined_matrix)

    # 5. Save in the exact schema dose_matrices() expects: row, col, value
    output_df = pd.DataFrame({
        'row':   coo.row,
        'col':   coo.col,
        'value': coo.data,
    })

    out_name = "Target.csv" if struct == "Prostate" else f"{struct}.csv"
    output_df.to_csv(out_name, index=False)

    print(f"  -> wrote {struct}.csv ({coo.nnz} non-zeros, "
          f"{combined_matrix.shape[0]} voxels x {combined_matrix.shape[1]} beamlets)")

In [ ]:
def get_structures(directory):
    '''
    Extracts all the csv files in the directory and returns a list containing the names of files
    '''
    structure_list = []
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            structure_list.append(filename)
    return structure_list

In [ ]:
def dose_matrices(structure_list, directory, beamlets):
    '''
    Gets the list of structures and the directory where the dosage files are located and
    returns a dictionary with the dosage matrices as values
    '''
    dose_mat = {}
    for structure in structure_list:
        struct_path = os.path.join(directory, structure)
        if os.path.exists(struct_path):
            dfCheck = pd.read_csv(struct_path, nrows=1)
            expected_columns = {'row', 'col', 'value'}
            if expected_columns.issubset(dfCheck.columns):
               df = pd.read_csv(struct_path)
            else:
                col_names = ["row","col","value"]
                df = pd.read_csv(struct_path, names=col_names, header=None)

            matrix = coo_matrix((df['value'], (df['row'], df['col'])))
            dose_mat[os.path.splitext(structure)[0]] = coo_matrix((df['value'], (df['row'], df['col'])), shape = (matrix.shape[0], beamlets))
        else:
            print(f"File {struct_path} does not exist in {directory}")
    return dose_mat

In [ ]:
def MILP(DoseMat, beamlets, Tmax, Tmin, M, UD, UV):

    # finite bound for beamlet intensity and ub-slack
    ub_beamlet = 10000
    ub_slack=10000

    #OAR structures
    oar_structs = []
    for struct in DoseMat.keys():
        if struct != "Target":
            oar_structs.append(struct)

    #OAR dimensions
    oar_dims = {}
    for s in oar_structs:
        oar_dims[s] = DoseMat[s].shape[0]

    #Target dimensions
    n_Target = DoseMat['Target'].shape[0]

    #Problem dimensions
    nbin = sum(oar_dims[s] for s in oar_structs) # number of binary variables
    n = nbin + beamlets + 2*n_Target # total number of variables

    # Indexes for each variable in decision vector
    oar_indices = {}
    index = 0
    for s in oar_structs:
        oar_indices[s] = index
        index += oar_dims[s]
    idx_x = nbin #index for beamlet variables
    idx_sl = nbin + beamlets #index for lower slack
    idx_su = nbin + beamlets + n_Target #index for upper slack

    # Q (zero matrix), w (objective: min sum(sl) + sum(su))
    Q = csr_matrix((n, n))
    w = np.zeros(n)
    w[idx_sl : idx_sl + n_Target] = 1.0  # sl coefficients
    w[idx_su : idx_su + n_Target] = 1.0  # su coefficients

    #Building A, b
    constraints = [] #A
    rhs = [] #b

    # Part 1: Target upper-dose - Dose*Intensity
    # overdose slack <= Tmax for each voxel in target
    TmaxArray = np.full(n_Target, Tmax) if np.isscalar(Tmax) else np.asarray(Tmax)
    part1 = hstack([
        csr_matrix((n_Target, nbin)), # zeros for binary
        DoseMat['Target'], # dose for each target voxel (D_Target @ x)
        csr_matrix((n_Target, n_Target)), #zeros for sl
        -eye(n_Target) #Identidy for su
        ])
    constraints.append(part1)
    rhs.append(TmaxArray)

    # Part 2: Target lower-dose
    # -Dose*Intensity - underdose slack <= -Tmin for each voxel in target
    #same idea as part 1
    TminArray = np.full(n_Target, Tmin) if np.isscalar(Tmin) else np.asarray(Tmin)
    part2 = hstack([
        csr_matrix((n_Target, nbin)),
        -DoseMat['Target'],
        -eye(n_Target),
        csr_matrix((n_Target, n_Target))
            ])
    constraints.append(part2)
    rhs.append(-TminArray)

    # Part 3: OAR dose constraints
    # Dose*intensity + (UD-M)*y <= UD
    for struct in oar_structs:
        n_struct = oar_dims[struct] # number of voxles (rows)
        struct_index = oar_indices[struct] # the column where the binary cols start for that struct

        #each struct needs zero-padding to take place for the other structs
        left_pad = csr_matrix((n_struct, struct_index))
        diag_block = (UD[struct] - M) * eye(n_struct)
        right_pad = csr_matrix((n_struct, nbin - struct_index - n_struct))

        #combine previous sections to create binary section of the constraint matrix
        binary_cols = hstack([left_pad, diag_block, right_pad])

        #Format binary, beamlets, sl, and su portion of the matrix
        block_oar = hstack([
            binary_cols,
            DoseMat[struct].tocsr(),
            csr_matrix((n_struct, n_Target)), #zero for sl
            csr_matrix((n_struct, n_Target)) #zero for su
        ])

        #set up rhs
        rhs_oar = np.full(n_struct, UD[struct]) if np.isscalar(UD[struct]) else np.asarray(UD[struct])

        #add to matrices
        constraints.append(block_oar)
        rhs.append(rhs_oar)

    # Block 4: Volume limit constraints — sum(y_S) <= UV_S * n_S
    for struct in oar_structs:
        n_struct = oar_dims[struct] #voxels per OAR
        struct_index = oar_indices[struct] # the column where the binary cols start for that struct #*****

        left_pad   = csr_matrix((1, struct_index)) #skip over potential different struct's values
        ones_block = csr_matrix(np.ones((1, n_struct))) #count voxels that exceed dose cap
        right_pad  = csr_matrix((1, n - struct_index - n_struct)) #skip all other rows

        #combine
        vol_row = hstack([left_pad, ones_block, right_pad])

        constraints.append(vol_row)
        rhs.append(np.array([UV[struct] * n_struct])) #max number of voxels allowed to exceed

    A = vstack(constraints)
    b = np.concatenate(rhs)

    # lower and upper bounds
    #lbs - y >= 0, x >= 0, sl >= 0, su >= 0
    lbs = np.concatenate((np.zeros(nbin), np.zeros(beamlets), np.zeros(n_Target), np.zeros(n_Target)))
    #ubs - y <= 1, x <= ub_beamlet, sl <= ub_slack, su <= ub_slack
    ubs = np.concatenate((np.ones(nbin), np.full(beamlets, ub_beamlet), np.full(n_Target, ub_slack), np.full(n_Target, ub_slack)))

    # and finally...
    prob = MIQP(Q, w, A=A, b=b, n_binary_vars=nbin, bounds=(lbs, ubs))

    return prob

In [ ]:
####Runner Part
patientID = 1
directory = os.getcwd()

structure_list = ["Bladder.csv", "Femur_Head_L.csv", "Femur_Head_R.csv", "Target.csv", "Rectum.csv"] # Option 1
#structure_list = get_structures(directory) # Option 2

beamlets = combined_matrix.shape[1]
DoseMat  = dose_matrices(structure_list, ("/content/OpenPhiSolve"), beamlets)

# Instance Parameters
Tmax = 72
Tmin = 70
M    = 72

UD = {}
UD['Femur_Head_L']      = 50
UD['Bladder']     = 40
UD['Rectum'] = 50
UD['Prostate']     = 40
UD['Femur_Head_R'] = 50

UV = {}
UV['Femur_Head_L'] = 0.2
UV['Bladder'] = 0.5
UV['Rectum'] = 0.3
UV['Prostate'] = 0.3
UV['Femur_Head_R'] = 0.2

In [ ]:
# Set up device; optional if device = 'cpu'
device = "cpu"
jax.config.update("jax_platforms", jax_device(device))

In [ ]:
# Construct the MIQP problem from IMPT parameters
prob = MILP(DoseMat, beamlets, Tmax, Tmin, M, UD, UV)

# Set up backend instance (QIHD)
n_shots = 100
n_steps = 10000
seed = 42
lc_pr = 10
slow_a = False
backend = QIHD(n_shots=n_shots, n_steps=n_steps, seed=seed, device=device, lc_pr=lc_pr, slow_a=slow_a)

# Set up refiner instance (PDQP)
iterations = 10000
refiner = PDQP(iterations=iterations, device=device)

# Solve the problem using PhiMIQP
model = PhiMIQP(prob, backend, refiner)
res = model.solve()

In [ ]:
def dvhPlotter(DoseMat, UD, UV, intensityVals, patientID):
    DenseDoseMat = {}
    for struct in list(DoseMat.keys()):
        DenseDoseMat[struct] = DoseMat[struct].todense()

    dvh = {}
    for struct in list(DenseDoseMat.keys()):
        dd = np.matmul(DenseDoseMat[struct],intensityVals)
        dvh_array = np.zeros((100,1))
        for i in range(100):
            dvh_array[i] = ((dd >= i).sum() / len(dd)) * 100
        dvh[struct] = dvh_array

    xax = np.linspace(0, 100, 100)

    plt.figure()
    ax = plt.gca()

    for struct in list(DenseDoseMat.keys()):
        plt.plot(xax, dvh[struct], label=struct)
        if struct != 'Target':
            plt.plot(UD[struct], UV[struct]*100, marker='x', label=f'{struct} DVH Point')

    plt.legend()

    ax.minorticks_on()
    ax.grid(which='minor', linestyle=':', linewidth='0.5')
    ax.grid(which='major', linestyle='-', linewidth='1')

    ax.xaxis.set_tick_params(which='both', top=True)
    ax.yaxis.set_tick_params(which='both', right=True)

    ax.spines['left'].set_position('zero')
    ax.spines['bottom'].set_position('zero')

    plt.xlabel('Dose')
    plt.ylabel('Volume (%)')

    plt.savefig(f'dvhCurve_{patientID}.png')

    plt.show()

In [ ]:
# Validate the solution and extract beamlet intensities
xs, cnts = res.refined_samples, res.sample_counts

objs = np.array([prob.obj(x) for x in xs])
maxvios = np.array([prob.max_vios(x) for x in xs])
feas = maxvios < 1e-4

minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
succ = objs[feas] <= minima + 1e-4
minimizer_vios = maxvios[feas][minimizer]
succ_prob = np.sum(cnts[feas][succ]) / n_shots
print(f"----- Running PhiSolve -----\nBackend: {backend.__class__.__name__}; Refiner: {refiner.__class__.__name__}.")
print(f"Incumbent minimum: {minima}, feasibility violations: {minimizer_vios}, success probability : {succ_prob}.")
#assert minima <= -1075 #Our minima wont be negative so idk what value to put here

# Get best beamlet intensities
best_z = xs[feas][minimizer] #sample with the smallest objective
nbin = prob.n_binary_vars # number of binary variables
intensityVals = np.maximum(best_z[nbin : nbin + beamlets].reshape(-1, 1), 0.0)

# Plot DVH curve
dvhPlotter(DoseMat, UD, UV, intensityVals, patientID)